# Lab 2 — Isaac Sim Development Workstation

**Purpose:** Deploy a GPU-powered remote desktop for visual development and debugging
of RL robot environments. Use this before Lab 4 (RL training) to validate that your
Isaac Lab environment works correctly.

**Read `BACKGROUND.md` first** for context on why this is needed and how it fits
into the overall pipeline.

---

## What this notebook does

| Section | What happens |
|---------|-------------|
| 0. Setup | Resolve AWS resources, check prerequisites |
| 1. Deploy workstation | Launch EC2 g6e.4xlarge with Isaac Sim via CDK |
| 2. Monitor deployment | Wait for bootstrap (~15 min), verify status |
| 3. Connect to DCV | Get the URL and login credentials |
| 4. Use Isaac Sim | Commands to run on the workstation |
| 5. Manage instance | Start / stop to control costs |
| 6. Teardown | Destroy when no longer needed |

## Cost
- ~**\$2.20/hr** while running (`g6e.4xlarge`)
- ~**\$41/month** for EBS storage (always-on)
- **Stop the instance when not debugging** — your work is preserved on EBS

## IAM Requirements
The role deploying CDK needs:
- `CloudFormationFullAccess` or admin
- `AmazonEC2FullAccess`
- `IAMFullAccess` (to create the instance role)


## Prerequisites & Known Issues

> **This notebook is not fully self-contained.** Read this section before running.

### External dependencies

| Dependency | Required? | Notes |
|-----------|-----------|-------|
| CDK (`cdk/`) | **Required** | The notebook calls `npx cdk deploy`. CDK must be bootstrapped. |
| Node.js ≥ 18 | **Required** | For CDK. Check: `node --version` |
| AWS Marketplace subscription | **Required** | Subscribe **before** deploying — see below. |
| SageMaker role with CDK IAM+S3 permissions | **Required** | See IAM section in README.md. |
| `physical-ai/isaac-lab` ECR image | Optional | Only for `run-isaac-lab.sh`. Built in Lab 4. |

### Marketplace subscription (one-time, do this first)

The EC2 instance uses the NVIDIA Isaac Sim Marketplace AMI. You must accept terms before deploying:

1. Go to: https://aws.amazon.com/marketplace/pp/prodview-bl35herdyozhw
2. Click **Continue to Subscribe** → **Accept Terms**
3. Wait ~30 seconds for the subscription to activate

Skipping this causes a `OptInRequired` error when CloudFormation tries to launch the EC2 instance.

### IAM permissions required on the deploying role

The default SageMaker execution role is missing two permission sets needed for CDK bootstrap.
Ask your AWS admin to attach these as inline policies (see `README.md` for the full JSON):

- **`CDKBootstrapIAM`** — `iam:CreateRole/DeleteRole/AttachRolePolicy/...` on `cdk-hnb659fds-*` roles
- **`CDKBootstrapS3`** — `s3:PutBucketPolicy/GetBucketPolicy/...` on `cdk-hnb659fds-assets-*` bucket

Without these, `cdk bootstrap` will fail with `AccessDenied` on IAM or S3 operations.

### Instance type

The Marketplace AMI **only supports `g6e` instances** (NVIDIA L40S). The `g5` family (A10G)
is not supported and will fail with "instance configuration not supported".
The default `INSTANCE_TYPE` below is already set correctly to `g6e.4xlarge`.

### CDK context flags

The deploy command includes `--context workstation=true` which is **required**.
Without it, CDK does not instantiate the workstation stack and deploy fails with
"No stacks match the name(s) PhysicalAi-dev-Workstation".

### Isaac Sim first launch

The first time you run `./run-isaac-sim-gui.sh`, Isaac Sim takes **5–10 minutes** to compile
shaders. If a dialog says "Isaac Sim is not responding", click **Wait** — it is still loading.


## 0 — Setup

In [21]:
import boto3, json, os, subprocess, sys, time
from pathlib import Path

REPO_ROOT = Path.cwd()
while REPO_ROOT.name != "aws-physical-ai-toolchain" and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent

REGION  = boto3.session.Session().region_name or "us-west-2"
ACCOUNT = boto3.client("sts", region_name=REGION).get_caller_identity()["Account"]
STACK_NAME = "PhysicalAi-dev-Workstation"

ec2 = boto3.client("ec2", region_name=REGION)
cf  = boto3.client("cloudformation", region_name=REGION)
ssm = boto3.client("ssm", region_name=REGION)

print(f"Region:  {REGION}")
print(f"Account: {ACCOUNT}")
print(f"Stack:   {STACK_NAME}")
print(f"CDK dir: {REPO_ROOT / 'cdk'}")


Region:  us-west-2
Account: YOUR_ACCOUNT_ID
Stack:   PhysicalAi-dev-Workstation
CDK dir: /home/sagemaker-user/aws-physical-ai-toolchain/cdk


In [22]:
# Check prerequisites
errors = []

# Node.js for CDK
result = subprocess.run(["node", "--version"], capture_output=True, text=True)
if result.returncode == 0:
    print(f"✅ Node.js: {result.stdout.strip()}")
else:
    errors.append("Node.js not found — install from https://nodejs.org")
    print("❌ Node.js not found — required for CDK")

# CDK
result = subprocess.run(["npx", "cdk", "--version"], capture_output=True, text=True,
                        cwd=str(REPO_ROOT / "cdk"))
if result.returncode == 0:
    print(f"✅ CDK: {result.stdout.strip()[:40]}")
else:
    errors.append("CDK not found — run: npm install -g aws-cdk")
    print("❌ CDK not found")

# g6e.4xlarge quota
try:
    limits = ec2.describe_instance_type_offerings(
        Filters=[{"Name": "instance-type", "Values": ["g6e.4xlarge"]}],
        LocationType="region"
    )
    if limits["InstanceTypeOfferings"]:
        print(f"✅ g6e.4xlarge available in {REGION}")
    else:
        print(f"⚠️  g6e.4xlarge not listed — check Service Quotas for GPU instances")
except Exception as e:
    print(f"⚠️  Could not verify g5 availability: {e}")

# Check if stack already deployed
try:
    stack = cf.describe_stacks(StackName=STACK_NAME)["Stacks"][0]
    print(f"✅ Stack already exists: {stack['StackStatus']}")
    STACK_EXISTS = True
except:
    print(f"ℹ️  Stack '{STACK_NAME}' not yet deployed")
    STACK_EXISTS = False

print()
if errors:
    print("Fix the issues above before proceeding.")
else:
    print("Prerequisites OK.")


✅ Node.js: v20.19.6
✅ CDK: 2.1125.0 (build 71fd29e)
✅ g5.4xlarge available in us-west-2
ℹ️  Stack 'PhysicalAi-dev-Workstation' not yet deployed

Prerequisites OK.


## 1 — Deploy the Workstation

Deploys a `g6e.4xlarge` EC2 instance with:
- Ubuntu 24.04 + NVIDIA driver
- NICE DCV remote desktop (browser access via port 8443)
- Docker + NVIDIA Container Toolkit
- Isaac Sim 5.1.0 installed in `~/IsaacSim`
- 512 GB encrypted EBS (work persists across stop/start)

**First-time deployment:** ~5 min for CloudFormation + ~15 min for UserData bootstrap.
The instance will reboot once to finalise the NVIDIA driver and desktop.

**Subsequent starts:** ~60 seconds.

> **VPN warning:** NICE DCV uses port 8443. If you're on corporate VPN, disconnect
> before opening the DCV URL — most VPNs block non-standard outbound ports.


In [31]:
# ── Configure deployment ─────────────────────────────────────────────────
# Your public IP — used to restrict DCV access to your machine only.
# Run: curl ifconfig.me   to find your current IP.
# Or set to "0.0.0.0/0" to allow access from anywhere (less secure).
MY_IP = ""   # ← paste your IP here, e.g. "203.0.113.42"

INSTANCE_TYPE = "g6e.4xlarge"   # 1x L40S 48GB, ~$2.20/hr (required by Marketplace AMI)
                                 # Alternative: "g6e.2xlarge" (~$1.65/hr, same GPU)

if MY_IP:
    ALLOWED_CIDR = f"{MY_IP}/32"
else:
    ALLOWED_CIDR = "0.0.0.0/0"
    print("ℹ️  No IP set — DCV will be open to the internet (auth still required)")

print(f"Instance type: {INSTANCE_TYPE}")
print(f"Allowed CIDR:  {ALLOWED_CIDR}")
print(f"CDK stack:     {STACK_NAME}")


ℹ️  No IP set — DCV will be open to the internet (auth still required)
Instance type: g6e.4xlarge
Allowed CIDR:  0.0.0.0/0
CDK stack:     PhysicalAi-dev-Workstation


In [32]:
# ── Install CDK dependencies (first time only) ───────────────────────────
result = subprocess.run(["npm", "install"], capture_output=True, text=True,
                        cwd=str(REPO_ROOT / "cdk"))
if result.returncode == 0:
    print("✅ CDK dependencies ready")
else:
    print("npm install failed:")
    print(result.stderr[-500:])


✅ CDK dependencies ready


In [34]:
# ── Bootstrap CDK (first time only — idempotent) ─────────────────────────
result = subprocess.run(
    ["npx", "cdk", "bootstrap", f"aws://{ACCOUNT}/{REGION}"],
    capture_output=True, text=True, cwd=str(REPO_ROOT / "cdk")
)
output = result.stdout + result.stderr
if result.returncode == 0 or "already bootstrapped" in output.lower() or "already exists" in output.lower():
    print("✅ CDK bootstrap ready")
else:
    print("Bootstrap output:")
    print(output[-500:])


✅ CDK bootstrap ready


In [35]:
# ── Deploy the workstation stack ─────────────────────────────────────────
# This takes ~5 min (CloudFormation creates the EC2 instance).
# The instance then bootstraps itself (~15 min) — monitor in Section 2.
print("Deploying workstation stack...")
print("(This creates the EC2 instance — ~5 min)")
print()

result = subprocess.run([
    "npx", "cdk", "deploy", STACK_NAME,
    "--context", "mode=simple",
    "--context", "workstation=true",
    "--context", f"allowedCidr={ALLOWED_CIDR}",
    "--context", f"instanceType={INSTANCE_TYPE}",
    "--require-approval", "never",
    "--outputs-file", "/tmp/workstation-outputs.json",
], text=True, cwd=str(REPO_ROOT / "cdk"))

if result.returncode == 0:
    outputs = json.loads(Path("/tmp/workstation-outputs.json").read_text())
    stack_outputs = outputs.get(STACK_NAME, {})
    print()
    print("✅ Stack deployed!")
    for k, v in stack_outputs.items():
        print(f"  {k}: {v[:100]}")
    INSTANCE_ID = None
    for k, v in stack_outputs.items():
        if "InstanceId" in k:
            INSTANCE_ID = v.strip()
    print(f"\nInstance ID: {INSTANCE_ID}")
    print("\nThe instance is now bootstrapping (~15 min).")
    print("Run Section 2 to monitor progress.")
else:
    print(f"Deploy failed (exit code {result.returncode})")
    print("Check the CloudFormation console for details.")


Deploying workstation stack...
(This creates the EC2 instance — ~5 min)


  Physical AI Toolchain
  Mode: simple  |  Env: dev  |  Edge: false  |  Batch: false
  Region: us-west-2




✨  Synthesis time: 5.68s

current credentials could not be used to assume 'arn:aws:iam::YOUR_ACCOUNT_ID:role/cdk-hnb659fds-file-publishing-role-YOUR_ACCOUNT_ID-us-west-2', but are for the right account. Proceeding anyway.
(node:3290344) Warning: NodeVersionSupportWarning: The AWS SDK for JavaScript (v3)
versions published after the first week of January 2027
will require node >=22. You are running node v20.19.6.

To continue receiving updates to AWS services, bug fixes,
and security updates please upgrade to node >=22.

More information can be found at: https://a.co/c895JFp
(Use `node --trace-warnings ...` to show where the warning was created)
current credentials could not be used to assume 'arn:aws:iam::YOUR_ACCOUNT_ID:role/cdk-hnb659fds-file-publishing-role-YOUR_ACCOUNT_ID-us-west-2', but are for the right account. Proceeding anyway.
current credentials could not be used to assume 'arn:aws:iam::YOUR_ACCOUNT_ID:role/cdk-hnb659fds-deploy-role-YOUR_ACCOUNT_ID-us-west-2', but are for t

arn:aws:cloudformation:us-west-2:YOUR_ACCOUNT_ID:stack/PhysicalAi-dev-Workstation/358fbaa0-7563-11f1-9660-0603c8bbc7af

✅ Stack deployed!
  DCVWebURL: Connect via: https://<PUBLIC_IP>:8443 (get IP from EC2 console or: aws ec2 describe-instances --inst
  WorkstationInstanceId: i-YOUR_INSTANCE_ID
  SSMConnect: aws ssm start-session --target i-YOUR_INSTANCE_ID
  SetPassword: aws ssm send-command --instance-ids i-YOUR_INSTANCE_ID --document-name "AWS-RunShellScript" --param
  Cost: ~$3.00/hr (g6e.4xlarge, 1x L40S GPU, on-demand us-west-2) + ~$41/mo for 512GB gp3 EBS. Actual cost v

Instance ID: i-YOUR_INSTANCE_ID

The instance is now bootstrapping (~15 min).
Run Section 2 to monitor progress.


## 2 — Monitor Bootstrap Progress

The UserData script runs automatically after the instance starts.
It installs the NVIDIA driver, NICE DCV, Docker, and Isaac Sim.
Takes ~15 minutes. The instance reboots once at the end.

Re-run the status cell until you see `Bootstrap complete`.


In [36]:
# ── Get instance ID if not already set ───────────────────────────────────
# Run this cell if INSTANCE_ID wasn't set in Section 1
# (e.g., if the stack was already deployed before)
try:
    stack = cf.describe_stacks(StackName=STACK_NAME)["Stacks"][0]
    for output in stack["Outputs"]:
        if "InstanceId" in output["OutputKey"]:
            INSTANCE_ID = output["OutputValue"].strip()
            break
    print(f"Instance ID: {INSTANCE_ID}")
    inst = ec2.describe_instances(InstanceIds=[INSTANCE_ID])["Reservations"][0]["Instances"][0]
    STATE = inst["State"]["Name"]
    PUBLIC_IP = inst.get("PublicIpAddress", "no IP yet")
    print(f"State:       {STATE}")
    print(f"Public IP:   {PUBLIC_IP}")
    print(f"Type:        {inst['InstanceType']}")
except Exception as e:
    print(f"Error: {e}")
    print("Make sure STACK_NAME is correct and the stack was deployed.")


Instance ID: i-YOUR_INSTANCE_ID
State:       running
Public IP:   44.247.229.133
Type:        g6e.4xlarge


In [37]:
# ── Check bootstrap progress via SSM ─────────────────────────────────────
# Re-run this cell to refresh. Wait for "Bootstrap complete" or "Isaac Sim ready".
try:
    result = ssm.send_command(
        InstanceIds=[INSTANCE_ID],
        DocumentName="AWS-RunShellScript",
        Parameters={"commands": [
            "tail -20 /var/log/workstation-bootstrap.log 2>/dev/null || echo 'Log not available yet'"
        ]},
    )
    cmd_id = result["Command"]["CommandId"]
    time.sleep(5)
    output = ssm.get_command_invocation(CommandId=cmd_id, InstanceId=INSTANCE_ID)
    print(output.get("StandardOutputContent", "No output yet"))
    if output.get("StandardErrorContent"):
        print("STDERR:", output["StandardErrorContent"][:200])
except Exception as e:
    print(f"SSM not ready yet ({e})")
    print("The instance may still be starting. Wait 2 min and retry.")


46b61b6d7239: Download complete
45f8eb5f3d64: Pull complete
7478e0ac0f23: Pull complete
217f19e84bb6: Pull complete
ccb3af6f7a2b: Pull complete
4ac634a388fa: Pull complete
7136bae5c750: Pull complete
10157deeab15: Pull complete
1564a0d327cb: Pull complete
27ce428b6e31: Pull complete
46b61b6d7239: Pull complete
89c456d67856: Pull complete
8393cc8da717: Download complete
3d4e27743c1f: Pull complete
66b1e8bbb8b7: Pull complete
91a68e7805a3: Pull complete
db4f85c0947f: Pull complete
f084456faee7: Pull complete
288bc0d973af: Pull complete
142ef6c70ab6: Pull complete



In [39]:
# ── Wait for bootstrap to complete ───────────────────────────────────────
# Re-run to check. When ready you'll see the DCV URL.
try:
    result = ssm.send_command(
        InstanceIds=[INSTANCE_ID],
        DocumentName="AWS-RunShellScript",
        Parameters={"commands": [
            "cat /var/log/workstation-bootstrap.summary 2>/dev/null && "
            "echo DCV_URL=https://$(curl -s http://169.254.169.254/latest/meta-data/public-ipv4):8443 || "
            "echo 'Bootstrap not complete yet'"
        ]},
    )
    cmd_id = result["Command"]["CommandId"]
    time.sleep(5)
    output = ssm.get_command_invocation(CommandId=cmd_id, InstanceId=INSTANCE_ID)
    print(output.get("StandardOutputContent", "No output"))
except Exception as e:
    print(f"({e}) — retry in 2 min")


Workstation bootstrap complete
DCV_URL=https://:8443



## 3 — Connect to the Workstation via DCV

Once the bootstrap is complete:

1. Get the DCV URL from the cell below
2. Open it in your browser
3. Accept the self-signed certificate warning (click "Advanced → Proceed anyway")
4. Login with username `ubuntu`, password `pai-lab1`
5. You'll see a full Ubuntu desktop with GPU acceleration

> **If you're on VPN:** Disconnect VPN first. Port 8443 is blocked by most corporate VPNs.
>
> **Password:** Change it immediately with `passwd` in a terminal on the workstation.


In [40]:
# ── Get DCV connection details ────────────────────────────────────────────
try:
    inst = ec2.describe_instances(InstanceIds=[INSTANCE_ID])["Reservations"][0]["Instances"][0]
    PUBLIC_IP = inst.get("PublicIpAddress", "not assigned yet")
    STATE     = inst["State"]["Name"]

    print(f"Instance state: {STATE}")
    print()
    if STATE == "running" and PUBLIC_IP != "not assigned yet":
        DCV_URL = f"https://{PUBLIC_IP}:8443"
        print(f"╔══════════════════════════════════════════════════════╗")
        print(f"║  DCV URL:  {DCV_URL:<42}║")
        print(f"║  Username: ubuntu                                    ║")
        print(f"║  Password: pai-lab1  (change with: passwd)           ║")
        print(f"╚══════════════════════════════════════════════════════╝")
        print()
        print("Open the URL in your browser (accept the cert warning).")
        print("Disconnect VPN first if on a corporate network.")
    elif STATE == "stopped":
        print("Instance is stopped. Run Section 5 to start it.")
    else:
        print(f"Instance is {STATE}. Wait and retry.")
except Exception as e:
    print(f"Error: {e}")


Instance state: running

╔══════════════════════════════════════════════════════╗
║  DCV URL:  https://44.247.229.133:8443               ║
║  Username: ubuntu                                    ║
║  Password: pai-lab1  (change with: passwd)           ║
╚══════════════════════════════════════════════════════╝

Open the URL in your browser (accept the cert warning).
Disconnect VPN first if on a corporate network.


## 4 — Using Isaac Sim on the Workstation

Once connected via DCV, open a terminal on the desktop and run these commands.

### Launch Isaac Sim GUI
```bash
source ~/IsaacSim/bin/activate
isaacsim
```
You'll see the full Isaac Sim 3D editor — robot loader, physics settings, scene tree.

### Run Isaac Lab training with visual rendering (to see robots training)
```bash
source ~/IsaacSim/bin/activate
cd /home/ubuntu/aws-physical-ai-toolchain
python3 -m isaaclab -p training/scripts/train.py \
    --task Isaac-Velocity-Flat-Anymal-D-v0 \
    --num_envs 16 \
    --max_iterations 10
```
Watch 16 simulated robots train in real time.

### Run the UR3 pick-and-place environment (visual validation)
```bash
source ~/IsaacSim/bin/activate
cd /home/ubuntu/aws-physical-ai-toolchain
python3 training/envs/pick_and_place_ur3.py --num_envs 4
```
This loads the UR3 environment defined in `training/envs/pick_and_place_ur3.py`.
Watch for physics issues, reward firing, object behaviour.

### Test the Isaac Lab training container (same as SageMaker)
```bash
# Pull your ECR container
ACCOUNT=$(aws sts get-caller-identity --query Account --output text)
REGION=$(aws configure get region)
aws ecr get-login-password --region $REGION | \
    docker login --username AWS --password-stdin $ACCOUNT.dkr.ecr.$REGION.amazonaws.com
docker pull $ACCOUNT.dkr.ecr.$REGION.amazonaws.com/physical-ai/isaac-lab:latest

# Run it (same as SageMaker would)
docker run --gpus all --rm \
    $ACCOUNT.dkr.ecr.$REGION.amazonaws.com/physical-ai/isaac-lab:latest
```

### What to look for when validating the RL environment
- Robot reaches toward the object (approach reward working)
- Gripper closes at the right time (grasp reward working)
- Object is lifted cleanly, not flung (success reward working)
- No objects clipping through the table (collision meshes correct)
- Domain randomisation looks reasonable (objects stay on table)


## 5 — Manage Instance (Start / Stop)

**Always stop the instance when you're done debugging.**
Your work is saved on the 512 GB EBS volume.
Restarting takes ~60 seconds.


In [ ]:
# ── Stop the instance (saves ~$1.62/hr) ──────────────────────────────────
# Uncomment and run when done with your session
ec2.stop_instances(InstanceIds=[INSTANCE_ID])
print(f"Stopping {INSTANCE_ID}... (your work is saved on EBS)")

In [44]:
# ── Start the instance when you need it again ─────────────────────────────
# ec2.start_instances(InstanceIds=[INSTANCE_ID])
# print(f"Starting {INSTANCE_ID}... (takes ~60s, then reconnect via DCV)")

# ── Check current state and cost ─────────────────────────────────────────
try:
    inst = ec2.describe_instances(InstanceIds=[INSTANCE_ID])["Reservations"][0]["Instances"][0]
    state = inst["State"]["Name"]
    itype = inst["InstanceType"]
    print(f"Instance {INSTANCE_ID}")
    print(f"  State: {state}")
    print(f"  Type:  {itype}")
    print()
    if state == "running":
        print("⚠️  Instance is RUNNING — charges accumulating (~$1.62/hr)")
        print("   Uncomment the stop line above when done.")
    elif state == "stopped":
        print("✅ Instance is STOPPED — no compute charges (EBS still ~$0.05/hr)")
    else:
        print(f"   State: {state}")
except Exception as e:
    print(f"Error: {e}")


Instance i-YOUR_INSTANCE_ID
  State: stopped
  Type:  g6e.4xlarge

✅ Instance is STOPPED — no compute charges (EBS still ~$0.05/hr)


## 6 — Teardown (optional)

Destroy the workstation when you no longer need it for development.  
**Warning:** This deletes the EBS volume and all work stored on it.

In [ ]:
# ── Destroy the workstation stack ────────────────────────────────────────
# WARNING: This deletes the EC2 instance AND the 512 GB EBS volume.
# All data on the instance will be lost. Only run if you're done with Lab 2.
#
# Uncomment to run:

# result = subprocess.run([
#     "npx", "cdk", "destroy", STACK_NAME,
#     "--force",
# ], text=True, cwd=str(REPO_ROOT / "cdk"))
#
# if result.returncode == 0:
#     print(f"✅ {STACK_NAME} destroyed — no more charges")
# else:
#     print(f"Destroy failed — check CloudFormation console")

print("Teardown cell is commented out for safety.")
print("Uncomment and run only when you're done with visual development.")


## 7 — Summary & Next Steps

### What you've done in Lab 2
- ✅ Deployed a GPU workstation with Isaac Sim and NICE DCV
- ✅ Connected via browser remote desktop
- ✅ Validated the RL environment visually
- ✅ Confirmed the training container works locally

### Next: Lab 4 — RL Refinement

With the environment validated, you can now run headless RL training at scale on
SageMaker. Lab 4 uses the same Isaac Lab environment but runs 4,096 parallel robot
copies on a `g5.12xlarge` instance, training much faster than visual mode.

```
Lab 1: GR00T fine-tuning (imitation)  → policy works ~70%
Lab 2: Visual debugging               → environment validated ✓
Lab 4: RL refinement (SageMaker)      → policy improves to ~95%
```

### Stop the instance before moving on
The instance costs ~$1.62/hr while running. Stop it from the cell above or the
EC2 console. You can restart it any time to continue visual development.
